# Early Detection of Collective Psychological Pressure

> **Aggregate, population-level language signals only. No diagnosis. No individual-level analysis.**

This notebook detects stress-related *linguistic signals* in public online text and reports only aggregate patterns
(cells with n < K_MIN are suppressed). It does not assess, profile, rank or re-identify any individual.
Findings are phrased as "stress-related language in the sampled communities shifted", never as "the population became stressed".

Training cells are marked `# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<` and run on Colab.

**Models go to `trained_models/` in the project root (no Google Drive).** Every training step is logged with timestamps in `trained_models/HANDOFF.md` and runs **once across both kernels**: a step already done on Colab is skipped here and in `notebook_local.ipynb` (the local-training twin of this notebook), and vice versa.

## Step 0: Project scaffold and environment

In [ ]:
# Step 0.0: COLAB SETUP. Run FIRST whenever the kernel is Colab (VS Code: Select Kernel > Colab). Local kernel: does nothing.
# Layout: everything lives on your PC. The Colab kernel only TRAINS (Step 3 cells marked USER RUNS). It gets the code +
# training data + the PC's handoff log from colab_bundle.zip on the SERVER's own temporary disk (build it locally with
# cell 3.0c, then in VS Code Explorer right-click data/outputs/colab_bundle.zip > "Upload to Colab"). No Google Drive:
# models go to /content/TA-BDA/trained_models on the server; cell 3G packs them into /content/colab_results.zip for the PC.
import os, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    PROJECT_DIR = Path("/content/TA-BDA")               # server disk, temporary

    def _find_bundle(base=Path("/content"), name="colab_bundle.zip", max_depth=3):
        for dirpath, dirnames, filenames in os.walk(base):
            depth = len(Path(dirpath).relative_to(base).parts)
            dirnames[:] = [d for d in dirnames if d not in ("drive", "sample_data", "TA-BDA") and depth < max_depth]
            if name in filenames:
                return Path(dirpath) / name
        return None

    bundle = _find_bundle()
    if bundle is not None:                              # (re)extract: a fresh upload always wins (overwrites code, no deletes;
        with zipfile.ZipFile(bundle) as z:              # trained_models/ is never in the bundle, so results are untouched)
            z.extractall(PROJECT_DIR)
        print("unpacked", bundle, "->", PROJECT_DIR)
    if not (PROJECT_DIR / "code" / "config.yaml").is_file():
        listing = sorted(p.name for p in Path("/content").iterdir())
        raise FileNotFoundError(
            f"colab_bundle.zip is not on THIS Colab server (/content holds: {listing}). 'Upload to Colab' writes to "
            "/content/colab_bundle.zip on the server you pick; if VS Code asked 'Select a server', pick the one this kernel "
            "uses (Colab panel > Contents shows each server's /content). Then re-run 0.0.")
    os.chdir(PROJECT_DIR / "code")
    import torch, transformers
    print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU:",
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (choose a GPU runtime)")
    print("models -> /content/TA-BDA/trained_models (server disk; pack with 3G) | checkpoints -> /content/stress_signals_checkpoints")
else:
    print("local kernel: Colab setup skipped")

In [ ]:
# Step 0.1: bootstrap (idempotent, safe to re-run). Imports the package, loads + validates config, seeds, versions.
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "code" / "config.yaml").is_file():
            return c
    raise FileNotFoundError("code/config.yaml not found above " + str(start))

PROJECT_ROOT = _find_root(Path.cwd().resolve())   # folder holding code/ and data/; all config paths resolve against it
SRC = str(PROJECT_ROOT / "code")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
# Only code/ goes on sys.path (it holds the stress_signals package); PROJECT_ROOT itself is never added.

from stress_signals import ETHICS_BANNER, __version__, get_logger, load_config, set_seed
from stress_signals.utils import library_versions, models_root
from stress_signals import handoff as H

CFG = load_config(PROJECT_ROOT / "code" / "config.yaml")
LOG = get_logger("stress_signals", CFG["logging"]["level"], CFG["_paths"]["logs"] / "pipeline.log")
SEED = CFG["primary_seed"]
set_seed(SEED, deterministic=CFG["deterministic"])

print("=" * 88)
print(ETHICS_BANNER)
print("=" * 88)
print(f"stress_signals {__version__} | project root: {PROJECT_ROOT}")
print(f"config: {CFG['_config_file']} | seeds: {CFG['seeds']} (primary {SEED}) | K_MIN: {CFG['privacy']['k_min']}")
print(f"models + handoff log: {models_root(CFG)}  (kernel: {H.kernel_name()})")
H.sync(CFG)                                      # Colab: merge the PC's handoff log from the bundle
print("library versions:")
for name, ver in library_versions().items():
    print(f"  {name:<13} {ver if ver else 'NOT INSTALLED'}")

In [ ]:
# Step 0.2: environment check (read-only; no model weights are created or updated)
from stress_signals.utils import detect_environment

ENV = detect_environment()
for key in ("python", "python_executable", "platform", "is_colab", "torch", "torch_cuda_build",
            "cuda_available", "arch_list", "gpu_name", "gpu_capability", "vram_total_gb", "vram_free_gb", "java"):
    print(f"{key:<18} {ENV[key]}")
print("sm_61 in arch list:", "sm_61" in ENV["arch_list"])
if ENV["warnings"]:
    print("\nWARNINGS:")
    for w in ENV["warnings"]:
        print("  !", w)
else:
    print("\nNo environment warnings.")

## Step 1: Acquire and audit all datasets

Read-only inspection: no model weights are created or updated. Outputs: `data/processed/_cache/audit_<name>.json|.md`,
`data/outputs/reports/data_audit.md`, `data/outputs/manifests/step1_audit_manifest.json`. No raw text is written to them.

**Kaggle credentials (GoEmotions Kaggle copy, optional):** kaggle.com → *Settings* → *API* → *Create New Token* downloads
`kaggle.json`. Save it as `%USERPROFILE%\.kaggle\kaggle.json` (Windows) or `~/.kaggle/kaggle.json` (Colab/Linux, then
`chmod 600`), or set the `KAGGLE_USERNAME` / `KAGGLE_KEY` environment variables. Never commit it. Without credentials the
audit records the Kaggle copy as `unavailable` and uses the official files plus the HF fallback.

In [ ]:
# Step 1.1: Zenodo header check FIRST (reads only the header row of teaching_post_features_tfidf_256.csv)
from stress_signals.utils import require_local_kernel; require_local_kernel("Step 1")
from stress_signals import audit as A

head = A.zenodo_header(CFG, "teaching", "post")
print("columns:", head["n_columns"])
print("non-feature columns:", head["non_feature_columns"])
for k in ("text_column_candidates", "author_columns", "date_columns", "id_columns"):
    print(f"{k:<24}", head[k] or "NONE")

In [ ]:
# Step 1.2: run all audits (idempotent: downloads are skipped when files exist; metadata responses are cached)
from stress_signals.utils import require_local_kernel; require_local_kernel("Step 1")
results = A.run(CFG)
for name, res in results.items():
    print(f"{name:<14}", "ERROR " + res["error"] if "error" in res else "ok", "| flags:", res.get("flags", []))
print("report:", CFG["_paths"]["reports"] / "data_audit.md")

In [ ]:
# Step 1.3: SenticNet eyeball view: LOCAL NOTEBOOK VIEW ONLY (scrubbed, truncated, 3 rows per file × label). Not saved anywhere.
from stress_signals.utils import require_local_kernel; require_local_kernel("Step 1")
import pandas as pd
from stress_signals.privacy import scrub_text

pd.set_option("display.max_colwidth", 220)
for name, df in A.load_senticnet(CFG).items():
    tcol, lcol = A.senticnet_text_label_cols(df)
    view = df.groupby(lcol, group_keys=False).apply(lambda g: g.sample(min(3, len(g)), random_state=SEED))
    view = view.assign(text=view[tcol].astype(str).map(lambda t: scrub_text(t, CFG)[:220]))[[lcol, "text"]]
    print(f"--- {name} (text column={tcol!r}, label column={lcol!r})")
    display(view)

In [ ]:
# Step 1.4: create the hand-check sheet (~50 rows stratified by file × label). Open it, fill `human_label` with 0/1
from stress_signals.utils import require_local_kernel; require_local_kernel("Step 1")
# using the definition "the text expresses stress", leave blank if unsure, save, then run Step 1.5.
sheet = A.make_senticnet_handcheck(CFG)   # will not overwrite an existing (possibly filled) sheet
print(sheet)

In [ ]:
# Step 1.5: score the hand-check (run after filling the sheet), then rebuild the report + manifest
from stress_signals.utils import require_local_kernel; require_local_kernel("Step 1")
hc = A.score_senticnet_handcheck(CFG)
print(hc)
A.audit_senticnet(CFG)
print(A.build_report(CFG))
A.write_audit_manifest(CFG);

## Step 2: Preprocessing, unified schema, splits

No model weights are created or updated here. Code: `code/stress_signals/preprocess.py`; tests: `code/tests/test_preprocess.py`.

**Unified schema** (first 8 columns of every `data/processed/*.parquet`): `record_id` (sha256 of source + native id),
`source`, `community`, `created_at` (UTC), `text_clean`, `n_chars`, `n_tokens_est` (regex word+punctuation count, *not* model
tokens), `lang`. Label/split columns follow. No author, username or URL fields exist anywhere.

**Decisions applied (approved in Step 1):**
- Dreaddit: exact-duplicate train texts dropped. If the copies disagree on the label, the whole group is dropped. Train texts that also appear in test are dropped. Official test = FINAL TEST.
  Train/validation comes from `StratifiedGroupKFold` on `post_id`. This is the stratified version of GroupShuffleSplit.
- GoEmotions: official files only, with the official train/dev/test splits (dev is renamed validation). Kaggle copy dropped.
- Zenodo pilot: only `subreddit, date, post` are read. `author` and the feature columns are never read. Text is deduplicated by hash across all files.
  The pre window really is 2018-11-01 to 2019-11-01. `created_at` is day-resolution only.
- SenticNet: validation-only parquet. Its labels are automated. Twitter_Non-Advert is deduplicated against Twitter_Full,
  rows that contain Dreaddit segments are removed, and duplicate groups with conflicting labels are dropped.
- SAD: cleaned into the unified schema. **No split here.** Step 5 owns the stressor splits and the Reddit gold set.

**Filters:** the language and minimum-length filters apply to unlabelled corpora only. Labelled benchmark rows, including official
test rows, are never dropped by them. `lang` is still recorded for them.
**Language ID: lingua-language-detector.** It is deterministic and offline (models ship in the wheel, which matters for Spark workers),
and its authors report better short-text accuracy than langdetect/langid/fastText (our check: UNVERIFIED).
**Chunking:** sentence-aligned windows of at most `max_length` tokens with a `stride`-token overlap. The bound is re-checked by re-tokenising.
**Aggregation:** the default is the *mean* of calibrated chunk probabilities. The max grows with the number of chunks, so long-post
communities (r/relationships) would look systematically more stress-laden than short-post ones. That would bias the aggregates this project reports.
The mean is length-neutral; `max` stays available for sensitivity analysis.


In [ ]:
# Step 2.1: cleaning demo on SYNTHETIC strings (no user text)
from stress_signals.utils import require_local_kernel; require_local_kernel("Step 2")
from collections import Counter
from stress_signals import preprocess as P

demo = [
    "OMG &amp;#x200B; **I'm SO stressed** about [rent](https://example.com/x) <url> 😭 ask u/someone or mail a.b@example.org",
    "> quoted\n\n# Heading\n* bullet\n[removed]",
    "Call (555) 123-4567 about the 2019 2020 2021 taxes...",
]
st = Counter()
for s in demo:
    c = P.clean_text(s, CFG, st)
    print(repr(c))
    print("   lexical:", P.clean_text_lexical(c, CFG, already_clean=True))
    assert P.clean_text(c, CFG) == c        # idempotent
print("edit counts:", dict(st))

In [ ]:
# Step 2.2: build all processed datasets + fixed splits + manifest (idempotent: existing outputs are skipped;
from stress_signals.utils import require_local_kernel; require_local_kernel("Step 2")
# P.run(CFG, force=True) rebuilds; saved splits are NEVER regenerated unless force_splits=True).
# Zenodo takes ~10-20 min the first time (per-file cache in data/processed/_cache/zenodo_clean/ makes it resumable).
import os
from stress_signals.utils import is_colab
if not is_colab():
    os.environ.setdefault("HF_HUB_OFFLINE", "1")   # local: Dreaddit is already in the HF cache since Step 1
RES2 = P.run(CFG)
for name, r in RES2.items():
    print(f"{name:<11} final_rows={r['final_rows']}")
print("report:", CFG["_paths"]["reports"] / "preprocess_report.md")

In [ ]:
# Step 2.3: split checks (aggregate counts only)
from stress_signals.utils import require_local_kernel; require_local_kernel("Step 2")
import pandas as pd

dre = pd.read_parquet(CFG["_paths"]["processed"] / "dreaddit.parquet")
print("Dreaddit split x label:\n", pd.crosstab(dre["split"], dre["label"], margins=True))
print("post_id overlap between splits (must all be 0):", P.group_overlap(dre))
print("official test rows all in FINAL TEST:", set(dre.loc[dre.official_split == "test", "split"]) == {"test"})

goe = pd.read_parquet(CFG["_paths"]["processed"] / "goemotions.parquet", columns=["split", "labels"])
print("GoEmotions split sizes:", goe["split"].value_counts().to_dict())
for name in ("sad", "zenodo_pilot", "senticnet_validation"):
    df = pd.read_parquet(CFG["_paths"]["processed"] / f"{name}.parquet", columns=["community", "lang", "n_tokens_est"])
    print(f"{name}: rows={len(df)}, lang={df['lang'].fillna('und').value_counts().head(3).to_dict()}, "
          f"median n_tokens_est={df['n_tokens_est'].median()}")

In [ ]:
# Step 2.4: chunking check on the Zenodo pilot: counts only, seeded sample, tokenizer of the stress model
from stress_signals.utils import require_local_kernel; require_local_kernel("Step 2")
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(CFG["models"]["stress"]["base_model"], use_fast=True)
ML, STRIDE, MAXC = CFG["models"]["stress"]["max_length"], CFG["chunking"]["stride"], CFG["chunking"]["max_chunks_per_doc"]
zp = pd.read_parquet(CFG["_paths"]["processed"] / "zenodo_pilot.parquet", columns=["community", "text_clean"])
sample = pd.concat([g.sample(min(len(g), 300), random_state=SEED) for _, g in zp.groupby("community")])
rows = []
for comm, t in zip(sample["community"], sample["text_clean"]):
    ch = P.chunk_text(t, tok, ML, STRIDE, MAXC)
    assert all(c.n_tokens <= ML for c in ch)
    rows.append((comm, ch[0].n_chunks_total, max(c.n_tokens for c in ch)))
cs = pd.DataFrame(rows, columns=["community", "n_chunks", "max_chunk_tokens"])
print(f"window={ML}, overlap={STRIDE}, max chunks kept={MAXC}; max tokens seen in any chunk: {cs.max_chunk_tokens.max()}")
print(cs.groupby("community")["n_chunks"].describe(percentiles=[.5, .9, .99]).round(2))
print("share of records needing >1 chunk:", round(float((cs.n_chunks > 1).mean()), 3),
      "| share with > max_chunks (subsampled):", round(float((cs.n_chunks > MAXC).mean()), 4))
print("aggregation example (mean of chunk probabilities):", P.aggregate_chunk_scores([0.2, 0.7, 0.6]))

In [ ]:
# Step 2.5: unit tests (cleaning idempotence, no PII left, chunk bound, no post_id overlap, split protection)
from stress_signals.utils import require_local_kernel; require_local_kernel("Step 2")
import subprocess, sys
from stress_signals.utils import is_colab
r = subprocess.run([sys.executable, "-m", "pytest", str(PROJECT_ROOT / "code" / "tests"), "-q", "-p", "no:cacheprovider",
                    "--rootdir", str(PROJECT_ROOT)], cwd=str(PROJECT_ROOT / "code"), capture_output=True, text=True,
                   env={**os.environ, "PYTHONPATH": SRC, **({} if is_colab() else {"HF_HUB_OFFLINE": "1"}), "PYTHONIOENCODING": "utf-8"})
print(r.stdout[-3000:], r.stderr[-2000:])

## Step 3: Stress-language classifier on Dreaddit

Code: `code/stress_signals/stress_model.py` (training, inference, shortcut checks, bundle), `code/stress_signals/metrics.py`
(metrics, bootstrap, calibration) and `code/stress_signals/handoff.py` (handoff log, run-once guard, Colab results transfer).
**Cells marked `USER RUNS THIS CELL` train models.** Run them on a Colab GPU here, or on the local kernel with `notebook_local.ipynb`.

**Where things are saved.** No Google Drive. Every model artefact goes to `trained_models/` in the project root on the kernel that ran it:
`E:\...\TA-BDA\trained_models` locally and `/content/TA-BDA/trained_models` on the Colab server, which is **temporary**. Cell **3G** packs the Colab side into
`/content/colab_results.zip`. Download that one file, then cell **3.0d** on the local kernel copies it into the project. It never overwrites files and never deletes anything.

**Run All is safe.** Training cells train only on the Colab kernel; on the local kernel they print `NOT RUN …` (use
`notebook_local.ipynb` to train here). Evaluation cells print `NOT RUN …` until every seed's results are on this kernel.

**Run once, across both kernels.** Every training step (baseline, each seed, each LOSO subreddit, the bundle) is logged in
`trained_models/handoff.json` / `HANDOFF.md` with UTC timestamps, the kernel, the duration and a fingerprint of the training data. A step logged as **done** on either kernel is
skipped by both notebooks. The PC's log travels to Colab inside `colab_bundle.zip`, and Colab's log comes back inside `colab_results.zip`.

| Kernel | Cells | Updates weights? |
|---|---|---|
| **local** | 0.0 (skips), 0.1, 3.0a, 3.0b, **3.0c** (build `colab_bundle.zip`, which includes the local handoff log) | no |
| **Colab GPU** (VS Code: Select Kernel → Colab) | upload `data/outputs/colab_bundle.zip` (right-click → *Upload to Colab*), then **0.0 → 0.1 → 3.0a → 3.1 → 3A.1 🔴 → 3B.1 🔴** (+ optional 3E.2 🔴) **→ 3G** | **yes** |
| *(transfer)* | download `/content/colab_results.zip` to the project root, `data/outputs/` or Downloads | no |
| **local** | 0.0 (skips), 0.1, 3.0a, **3.0d** (import), then 3A.2, 3B.2, 3C.*, 3D.*, 3E.1, 3E.3, 3F | no (3E.1 runs inference) |

Epoch checkpoints live on the Colab server's own disk, outside `trained_models/`, because the Trainer deletes old ones. After a *disconnect*,
re-run 0.0 → 0.1 → 3.0a and the training cell: finished seeds are skipped and an unfinished seed resumes. If the server was *recycled*, everything on it is gone
(run 3G after each finished seed to be safe). Re-upload the bundle. Seeds whose results you already imported on the PC are skipped, because the bundle carries the PC's log.

**Decisions.**
- Input is `text_clean` (the Step 2 cleaning of Dreaddit `text`) only. The LIWC/DAL/syntax/social columns are dropped explicitly.
- Model selection uses validation `f1_stress`. The test set is never used for selection, calibration or thresholds.
- `confidence` sample weights are off by default; the docstring of `confidence_weights` explains why.
- The deployable model is the seed with the best validation F1(stress). Temperature and threshold are fitted on validation.
  The threshold is searched only in 0.40-0.60, maximising validation F1(stress). The tuned threshold is **applied only if the validation gain over 0.5 is clear**:
  gain >= 0.01 and the 95% CI of the paired cluster-bootstrap gain is above 0. Otherwise 0.5 is used. Test metrics are reported at **both** 0.5 and the
  tuned threshold, once (guarded file).

In [ ]:
# Step 3.0a: settings, transformers argument names, data guard (no training; runs locally or on Colab)
from stress_signals import stress_model as SM, metrics as MX

S3 = SM.settings(CFG)                       # pass a model name to switch, e.g. SM.settings(CFG, "distilroberta-base")
compat = SM.hf_compat()
print(f"model: {S3['model_choice']} -> {S3['hf_id']} | max_len {S3['max_length']} | lr {S3['learning_rate']} | "
      f"batch {S3['train_batch_size']} | epochs {S3['epochs']} | warmup {S3['warmup_ratio']} | seeds {S3['seeds']}")
print(f"transformers {compat['transformers']}: eval key = {compat['eval_strategy_key']!r}, "
      f"Trainer tokenizer arg = {compat['tokenizer_key']!r}")
D3 = SM.load_dreaddit(CFG)                  # verifies the saved split file and keeps text-only columns
for k, v in D3.items():
    print(f"{k:<10} n={len(v):>5}  stress share={v['label'].mean():.3f}  columns={list(v.columns)}")
print("runs will be written to:", SM.runs_root(CFG))

In [ ]:
# Step 3.0b: LABEL MEANING CHECK. Local view only: scrubbed, truncated, 3 per label, never saved.
# The HF card only says `label: int64`. Confirm by reading: do label-1 texts express stress?
import pandas as pd
pd.set_option("display.max_colwidth", 200)
view = (D3["train"].groupby("label", group_keys=False)
        .apply(lambda g: g.sample(3, random_state=SEED))[["label", "community", "text_clean"]])
view["text_clean"] = view["text_clean"].str.slice(0, 200)
display(view)

In [ ]:
# Step 3.0c (LOCAL kernel): build the Colab upload bundle (code/ + Dreaddit parquet + split file, ~1 MB).
from stress_signals.utils import require_local_kernel; require_local_kernel("Cell 3.0c (bundle builder)")
# Re-run after ANY code change AND after local training (it carries the local handoff log, so Colab skips what the PC
# already did), then re-upload it (VS Code Explorer: right-click the file > "Upload to Colab") and re-run 0.0.
from stress_signals.utils import build_colab_bundle
B = build_colab_bundle(CFG)
print(B, f"{B.stat().st_size/1e6:.2f} MB")

In [ ]:
# Step 3.0d (LOCAL kernel): bring in what Colab trained, then show the handoff log (what ran where, and when).
from stress_signals.utils import require_local_kernel; require_local_kernel("Import of colab_results.zip")
# Looks for colab_results*.zip in the project root, data/outputs/ and your Downloads folder (newest wins). Existing files
# are never overwritten and nothing is deleted; the Colab log is merged. Importing the same zip twice does nothing.
H.import_colab_results(CFG)          # or: H.import_colab_results(CFG, r"C:\path\to\colab_results.zip")
H.show(CFG)

In [ ]:
# Step 3.1: GPU check before training (Colab setup itself is cell 0.0)
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU: Runtime > Change runtime type > T4 GPU")
# deberta-v3-base additionally needs: !pip -q install sentencepiece

### 3A. Baseline: TF-IDF (word 1-2 + char_wb 2-5) with logistic regression
This is a reference point. The transformer has to beat it by a clear margin, beyond the confidence intervals, to justify its cost. C is chosen on validation F1(stress).

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 3A.1: fit the baseline (CPU is fine, ~1-3 min). Runs ONCE across both kernels: skipped when the handoff log says
# it is done (here, on the PC or on Colab) or when runs/baseline_tfidf_lr/DONE.json exists.
import pandas as pd
BASE_DIR = SM.runs_root(CFG) / "baseline_tfidf_lr"
H.run_step(CFG, H.BASELINE_STEP, "Fit TF-IDF + logistic regression baseline", lambda: SM.fit_baseline(CFG, D3),
           done_marker=BASE_DIR / "DONE.json", skip_result=BASE_DIR,
           allow=H.kernel_name() == "colab", blocked_reason=H.COLAB_ONLY)
if (BASE_DIR / "c_selection_validation.csv").exists():
    print(pd.read_csv(BASE_DIR / "c_selection_validation.csv").round(4))

In [ ]:
# Step 3A.2: baseline metrics (validation + test, threshold 0.5, cluster-bootstrap 95% CIs). Reads saved predictions.
R3 = SM.readiness(CFG)
if H.ready(R3["baseline"], "3A.2", R3["why_baseline"]):
    BASE_EVAL = SM.evaluate_baseline(CFG)
    for split in ("validation", "test"):
        m, ci = BASE_EVAL[split]["all"], BASE_EVAL[split]["ci"]
        print(f"{split:<10} " + "  ".join(f"{k}={m[k]:.3f} [{ci[k]['lo']:.3f},{ci[k]['hi']:.3f}]"
                                          for k in ("f1_stress", "f1_macro", "roc_auc", "pr_auc", "accuracy")))
    H.log_ran(CFG, "3A.2/baseline_eval", "Evaluate baseline")

### 3B. Transformer fine-tuning (Colab GPU)
There is one run per seed, and each is resume-safe. With roberta-base on a T4 GPU, expect a few minutes per epoch. Switch models with `MODEL_CHOICE`.

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
import json
# Step 3B.1: fine-tune all seeds. Each seed runs ONCE across both kernels: a seed the handoff log marks done (here, on
# the PC or on Colab) is skipped. Re-running after a disconnect resumes the unfinished seed from its last checkpoint.
MODEL_CHOICE = CFG["stress_model"]["model_choice"]     # "roberta-base" | "distilroberta-base" | "deberta-v3-base"
for seed in CFG["stress_model"]["seeds"]:
    rd = SM.run_dir_for(CFG, MODEL_CHOICE, seed)
    H.run_step(CFG, H.train_step_id(MODEL_CHOICE, seed), f"Fine-tune {MODEL_CHOICE}, seed {seed}",
               lambda seed=seed: SM.train_stress_seed(CFG, seed, MODEL_CHOICE), done_marker=rd / "DONE.json", skip_result=rd,
               allow=H.kernel_name() == "colab", blocked_reason=H.COLAB_ONLY)
    if (rd / "train_summary.json").exists():
        summ = json.loads((rd / "train_summary.json").read_text())
        print(f"seed {seed}: {rd} | best validation f1_stress = {summ.get('best_metric_validation_f1_stress')} "
              f"| epochs = {summ.get('epochs_completed')}")

In [ ]:
# Step 3B.2: training curves from the CSV logs (no training). Validation F1(stress) per epoch, per seed.
import json
MODEL_CHOICE = CFG["stress_model"]["model_choice"]
for seed, rd in SM.completed_seeds(CFG, MODEL_CHOICE).items():
    log = pd.read_csv(rd / "train_log.csv")
    ev = log[log["key"].isin(["eval_f1_stress", "eval_loss"])].pivot_table(index="epoch", columns="key", values="value")
    print(f"seed {seed}\n", ev.round(4))

### 3C. Evaluation (after training): validation and official test, uncalibrated at threshold 0.5
CIs come from a **cluster bootstrap over post_id** (1,000 resamples), because segments of the same post are not independent. *High-confidence* means
Dreaddit `confidence >= stress_model.high_confidence_threshold`. Reading these numbers: if the high-confidence F1 is much higher, many of the errors fall on
texts that annotators also disagreed about.

In [ ]:
# Step 3C.1: per-seed metrics + CIs, mean +/- std across seeds (reads saved logits; no training)
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3C.1", R3["why_seeds"]):
    EV3 = SM.evaluate_runs(CFG, MODEL_CHOICE)
    KEYS = ["accuracy", "precision_stress", "recall_stress", "f1_stress", "f1_macro", "roc_auc", "pr_auc"]
    for split in ("validation", "test"):
        rows = []
        for seed, r in EV3["seeds"].items():
            m, ci = r[split]["all"], r[split]["ci"]
            rows.append({"seed": seed, **{k: f"{m[k]:.3f} [{ci[k]['lo']:.3f},{ci[k]['hi']:.3f}]" for k in KEYS},
                         "confusion": m["confusion"]})
        print(f"\n== {split} (threshold 0.5, uncalibrated)"); display(pd.DataFrame(rows))
        print("mean +/- std across seeds:", {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in EV3["summary"][split].items() if k in KEYS})
    H.log_ran(CFG, "3C.1/evaluate_seeds", "Evaluate all seeds (uncalibrated)")

In [ ]:
# Step 3C.2: high-confidence subset (same metrics restricted to confident annotations)
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3C.2", R3["why_seeds"]):
    for split in ("validation", "test"):
        hc = {seed: r[split]["high_confidence"] for seed, r in EV3["seeds"].items()}
        first = next(iter(hc.values()))
        print(f"{split}: confidence >= {first['confidence_threshold']} keeps {first['share_of_split']:.1%} of rows (n={first['n']})")
        print("  mean +/- std:", {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in EV3['summary'][f'{split}_high_confidence'].items() if k in KEYS})

In [ ]:
# Step 3C.3: per-subreddit error analysis (aggregates) + a small LOCAL error view (scrubbed, truncated, never saved)
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3C.3", R3["why_seeds"]):
    FINAL_SEED = SM.select_final_seed(EV3)            # best VALIDATION f1_stress
    print("final seed (best validation f1_stress):", FINAL_SEED)
    display(SM.per_subreddit_errors(CFG, FINAL_SEED, "test", MODEL_CHOICE).round(3))
    from stress_signals.privacy import scrub_text
    te = SM.load_predictions(SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED), "test").merge(
        D3["test"][["record_id", "text_clean"]], on="record_id")
    te["error"] = (te["p"] >= 0.5).astype(int) != te["label"]
    errs = te[te["error"]].assign(margin=lambda d: (d["p"] - 0.5).abs()).sort_values("margin", ascending=False)
    nshow = min(CFG["privacy"]["max_error_examples_in_notebook"], 10)
    view = errs.head(nshow)[["community", "label", "p", "confidence"]].assign(
        text=errs.head(nshow)["text_clean"].map(lambda t: scrub_text(t, CFG)[:180]))
    display(view)   # most confident mistakes: label noise, sarcasm, or topic shortcuts?

### 3D. Calibration and threshold (fitted on VALIDATION only), then the calibrated test report once
- **Temperature scaling** fits one scalar T that minimises validation NLL. It changes confidence but not the ranking, so ROC-AUC is unchanged.
- **Threshold:** search 0.40-0.60 for the best validation F1(stress). Apply it only if the gain over 0.5 is clear (gain >= `apply_rule.min_gain`
  and the paired-bootstrap CI is above 0). The gain is measured on the same validation set that chose the threshold, so it is optimistic; the minimum gain is the safety margin.
- **Reading ECE:** it is the average gap between predicted probability and observed frequency, so lower is better. Downstream, aggregate signals average calibrated probabilities,
  so calibration matters more for this project than the last point of F1.

In [ ]:
# Step 3D.1: fit T + threshold on validation; reliability diagram; ECE / Brier before vs after (validation)
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3D.1", R3["why_seeds"]):
    CAL3 = SM.calibrate(CFG, FINAL_SEED, MODEL_CHOICE)
    dec = CAL3["threshold_decision"]; g = dec["validation_gain_f1_stress"]
    print(f"T = {CAL3['temperature']:.3f} | tuned threshold (searched {CAL3['threshold_search_grid'][:2]}) = {CAL3['threshold_tuned']:.2f}")
    print(f"validation F1(stress) gain over 0.5 = {g['gain']:+.4f} [95% CI {g['lo']:+.4f}, {g['hi']:+.4f}] "
          f"-> APPLIED threshold = {CAL3['threshold']:.2f} ({dec['reason']})")
    for k in ("before", "after"):
        v = CAL3[f"validation_{k}"]
        print(f"validation {k:<6}: ECE {v['ece']:.4f} | Brier {v['brier']:.4f} | F1(stress) {v['f1_stress']:.3f} @ {v['threshold']:.2f}")
    print(f"validation after, at tuned {CAL3['threshold_tuned']:.2f}: F1(stress) {CAL3['validation_after_tuned']['f1_stress']:.3f}")
    from IPython.display import Image
    display(Image(str(SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "reliability_validation.png")))
    H.log_ran(CFG, "3D.1/calibrate", "Calibrate final seed on validation")

In [ ]:
# Step 3D.2: FINAL calibrated test report: computed ONCE (FINAL_TEST_REPORT.json is reused afterwards).
# Reported at BOTH thresholds; "applied" is the one the decision rule picked on validation.
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3D.2", R3["why_seeds"]):
    FIN3 = SM.final_test_report(CFG, FINAL_SEED, MODEL_CHOICE)
    print(f"TEST (T={FIN3['temperature']:.3f}); applied threshold = {FIN3['threshold']:.2f} [{FIN3['threshold_applied']}]: "
          f"{FIN3['threshold_decision_reason']}")
    rows = []
    for name in ("at_threshold_0.5", "at_threshold_tuned"):
        b = FIN3[name]
        for k in KEYS + ["brier"]:
            rows.append({"threshold": f"{b['threshold']:.2f}", "metric": k, "value": round(b["test"][k], 3),
                         "ci95": f"[{b['test_ci'][k]['lo']:.3f}, {b['test_ci'][k]['hi']:.3f}]" if k in b["test_ci"] else ""})
    display(pd.DataFrame(rows).pivot(index="metric", columns="threshold", values=["value", "ci95"]))
    print("n =", FIN3["test"]["n"], "| ECE (threshold-free)", round(FIN3["test_ece"], 4))
    for name in ("at_threshold_0.5", "at_threshold_tuned"):
        b = FIN3[name]
        print(f"  @{b['threshold']:.2f} confusion {b['test']['confusion']} | high-confidence F1(stress) "
              f"{b['test_high_confidence']['f1_stress']:.3f} (n={b['test_high_confidence']['n']})")
    H.log_ran(CFG, "3D.2/final_test_report", "Final calibrated test report")

### 3E. Shortcut-learning checks
**(i) Topic masking (inference only).** Subreddit names and domain words (money, rent, abuse, boyfriend, ...) are replaced with the mask token.
A *control* condition masks the same number of random other words in each text, repeated over **5 random draws** (mean, sd and range are reported). How to read it:
- If the topic-masking drop is larger than **every** control draw's drop (`topic_drop_exceeds_all_control_draws`), the model leans on topic words, and its scores may reflect *what a
  community talks about* rather than how stressed the language is. That is a direct threat to comparisons between communities.
- If both drops are similar and small, the model mostly uses stress expression.
- Some drop is expected, because words like "anxiety" also carry real stress signal.

**(ii) Leave-one-subreddit-out (training, optional).** Train without one subreddit and test on all of its rows. How to read it: a large gap compared with in-domain
performance means the model does not generalise to unseen communities. That matters because the target corpus has communities Dreaddit lacks.

**(iii) Baseline top n-grams.** Look at the words with the largest weights. If the strongest stress words are topic words (`topic_term=True`) rather than
affect, coping or pressure words ("can't", "anxious", "scared", "overwhelmed"), the baseline is keying on topic.

In [ ]:
# Step 3E.1: topic masking vs random-masking control (5 draws) on the official test set (INFERENCE ONLY; GPU or CPU)
# Uses the APPLIED threshold from 3D. Diagnostic only: nothing is tuned on these numbers.
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3E.1", R3["why_seeds"]):
    model_dir = SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "best_model"
    SHORT = SM.shortcut_masking_eval(CFG, model_dir, "test", CAL3["temperature"], CAL3["threshold"], MODEL_CHOICE)
    display(SHORT[["condition", "draw_seed", "f1_stress", "roc_auc", "accuracy", "predicted_positive_rate"]].round(3))
    print(f"texts with >=1 masked term: {SHORT['texts_with_masked_terms'].iloc[0]} of {SHORT['n'].iloc[0]}; "
          f"terms masked: {SHORT['masked_terms_total'].iloc[0]}")
    display(SM.summarise_masking(SHORT).round(4))
    H.log_ran(CFG, "3E.1/topic_masking", "Topic-masking shortcut check")

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 3E.2 (OPTIONAL): leave-one-subreddit-out. Trains one model per held-out subreddit (distilroberta by default,
# 1 seed, config stress_model.shortcut.loso_*). ~10 trainings: run on a GPU. Each subreddit is one handoff step, so it
# runs ONCE across both kernels; resume-safe like 3B.1.
import pandas as pd
LOSO_MODEL = CFG["stress_model"]["shortcut"]["loso_model"]
for sub in sorted(pd.concat(D3.values())["community"].unique()):
    d = SM.runs_root(CFG) / "loso" / f"{LOSO_MODEL}_{sub}"
    H.run_step(CFG, H.loso_step_id(LOSO_MODEL, sub), f"LOSO: train {LOSO_MODEL} without r/{sub}",
               lambda sub=sub: SM.train_loso(CFG, [sub])[sub], done_marker=d / "DONE.json", skip_result=d,
               allow=H.kernel_name() == "colab", blocked_reason=H.COLAB_ONLY)
LOSO = SM.evaluate_loso(CFG)
if not len(LOSO):
    print("no finished LOSO runs on this kernel yet")
elif "FINAL_SEED" in globals():                        # in-domain comparison needs 3C.3 (local evaluation)
    inorig = SM.per_subreddit_errors(CFG, FINAL_SEED, "test", MODEL_CHOICE)[["community", "n", "f1_stress", "roc_auc"]]
    display(LOSO.merge(inorig, left_on="heldout_subreddit", right_on="community", suffixes=("_heldout", "_in_domain_test")).round(3))
else:
    display(LOSO.round(3))

In [ ]:
# Step 3E.3: baseline n-grams with the largest weights (needs 3A.1). topic_term=True = contains a subreddit/domain term.
R3 = SM.readiness(CFG)
if H.ready(R3["baseline"], "3E.3", R3["why_baseline"]):
    NG = SM.top_ngrams(CFG)
    for direction in ("stress", "not_stress"):
        sub = NG[(NG["direction"] == direction) & (NG["type"] == "word")]
        print(f"\n{direction}: share of top word n-grams that are topic terms = {sub['topic_term'].mean():.2f}")
        display(sub[["rank", "ngram", "coef", "topic_term"]].head(20))

### 3F. Save the model bundle
This writes `trained_models/stress/v<YYYYMMDD>/`. The bundle holds the weights and tokenizer,
`stress_config.json` (label map, max length, chunking rule, temperature, threshold, model name, seed), `metrics.json`, `model_card.md` and
`manifest.json` (SHA-256 of every file). It refuses to overwrite an existing bundle, and the handoff log makes sure it is written only once across both kernels.

In [ ]:
# Step 3F: write the bundle (no training), ONCE across both kernels. Run after 3D.2 (and optionally 3A, 3E.1, 3E.2 so
# metrics.json includes them). Output: trained_models/stress/v<YYYYMMDD>/.
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"] and "FINAL_SEED" in globals() and (SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "FINAL_TEST_REPORT.json").exists(), "3F", R3["why_seeds"]):
    BUNDLE3 = H.run_step(CFG, H.BUNDLE_STEP, f"Save stress model bundle ({MODEL_CHOICE}, seed {FINAL_SEED})",
                         lambda: SM.save_bundle(CFG, FINAL_SEED, MODEL_CHOICE))
    if BUNDLE3 is None:                                    # skipped: show the bundle the log points to
        ev = H.done_event(CFG, H.BUNDLE_STEP)
        BUNDLE3 = PROJECT_ROOT / ev["outputs"][0] if ev and ev["outputs"] else None
    print("bundle:", BUNDLE3)
    if BUNDLE3 is not None and BUNDLE3.is_dir():
        for f in sorted(BUNDLE3.iterdir()):
            print(f"  {f.name:<28} {f.stat().st_size/1e6:8.2f} MB")
        print((BUNDLE3 / "model_card.md").read_text()[:1500])

### 3G. Send Colab results to the PC (Colab kernel)
Run after each training session on Colab, ideally after every finished seed, because the server disk is temporary. It packs `/content/TA-BDA/trained_models`
(runs, predictions, logs and the handoff log) into **`/content/colab_results.zip`**. LOSO weights are left out by default because evaluation only needs their predictions.
Download that file. In VS Code, use the Colab server's file view (*Colab* panel → *Contents*) → right-click → *Download*; in browser Colab, use *Files* → ⋮ → *Download*.
Put it in the project root, `data/outputs/` or your Downloads folder. Then run **3.0d** on the local kernel.

In [ ]:
# Step 3G (COLAB kernel): pack everything trained on this server into ONE file for the PC.
from stress_signals.utils import is_colab
if is_colab():
    RESULTS_ZIP = H.pack_colab_results(CFG)            # include_loso_weights=True also ships LOSO weights (~330 MB each)
    print("download", RESULTS_ZIP, "then run 3.0d on the LOCAL kernel")
    H.show(CFG)
else:
    print("local kernel: nothing to pack; local results are already in", H.models_root(CFG))

## Step 4: Emotion model on GoEmotions

_Not implemented yet. Filled in at Step 4._

## Step 5: Stressor taxonomy and stressor classifier

_Not implemented yet. Filled in at Step 5._

## Step 6: TensiStrength auxiliary signal and SenticNet external validation

_Not implemented yet. Filled in at Step 6._

## Step 7: Unified inference package and artifact bundle

_Not implemented yet. Filled in at Step 7._

## Step 8: Ingestion to a Parquet data lake (pilot, then main corpus)

_Not implemented yet. Filled in at Step 8._

## Step 9: PySpark distributed NLP inference

_Not implemented yet. Filled in at Step 9._

## Step 10: Spark analytics (temporal, emotions, stressors, topics)

_Not implemented yet. Filled in at Step 10._

## Step 11: Baseline and change detection -> collective pressure signals

_Not implemented yet. Filled in at Step 11._

## Step 12: PostgreSQL storage, aggregates only (optional)

_Not implemented yet. Filled in at Step 12._

## Step 13: Streamlit dashboard

_Not implemented yet. Filled in at Step 13._

## Step 14: Final evaluation report, ethics and documentation

_Not implemented yet. Filled in at Step 14._

## Step 15: Kafka + Spark Structured Streaming (optional)

_Not implemented yet. Filled in at Step 15._